# 0 · Data Check — HH 数据规模盘点

目的：在彻底重做前后，盘点 **HH** 这批数据的规模，建立 before → after 对照。

- **旧产物（重做前）**：`optimized_HH_chunks/`、`optimized_HH_embeddings/`
- **新链路输入**：`converted_raw_txts/`（已含 HH + Qs）
- **新链路产出（分块后）**：重做表里 HH 子集的 chunk 数 + volume —— 待 `1_build_chunks` 跑完回来跑

> 本 repo 需作为 **Databricks Repo** 使用：repo 根目录会自动加入 `sys.path`，可直接 `from src... import`。

In [ ]:
from src import config as C
from src.stats import fs_stats, jsonl_chunk_stats, count_doc_folders, table_stats, bytes_human

## 1) HH 旧产物 · optimized_HH_chunks（原始文件数 + size）

In [ ]:
# HH legacy chunks: doc folders, chunk records, raw jsonl volume
hh_docs   = count_doc_folders(spark, C.HH_OPT_CHUNKS)
hh_chunks = jsonl_chunk_stats(spark, C.HH_OPT_CHUNKS + "*/chunks.jsonl")
hh_vol    = fs_stats(spark, C.HH_OPT_CHUNKS, glob="*.jsonl", recursive=True)

print("optimized_HH_chunks")
print("  doc folders   :", hh_docs)
print("  chunk records :", hh_chunks["n_chunks"])
print("  source docs   :", hh_chunks["n_docs"])
print("  jsonl files   :", hh_vol["n_files"])
print("  volume        :", bytes_human(hh_vol["total_bytes"]))

## 2) HH 旧产物 · optimized_HH_embeddings（文件数 + size）

In [ ]:
hh_emb = fs_stats(spark, C.HH_OPT_EMB, glob="*", recursive=True)
print("optimized_HH_embeddings")
print("  files  :", hh_emb["n_files"])
print("  volume :", bytes_human(hh_emb["total_bytes"]))

## 3) 新链路输入 · converted_raw_txts（全量，含 HH + Qs）

In [ ]:
src_all = fs_stats(spark, C.SRC_TXT_DIR, glob="*.txt", recursive=True)
print("converted_raw_txts (ALL)")
print("  txt files :", src_all["n_files"])
print("  volume    :", bytes_human(src_all["total_bytes"]))

## 4) 新链路产出（分块后）— 待 `1_build_chunks` 跑完回来跑

统计重做后 **HH 子集** 的 chunk 数与 volume，用于和上面旧 `optimized_HH` 对照。
`HH_TAG` 是 HH 文件在 `source_file`/路径中的标识子串，目前为占位，确认后调整。

In [ ]:
from pyspark.sql import functions as F

HH_TAG = "HH"  # TODO: confirm how HH files are identified in source_file / path

# Run AFTER 1_build_chunks has written C.TBL_CHUNKS
chunks = spark.table(C.TBL_CHUNKS)
hh_new = chunks.filter(F.col("source_file").contains(HH_TAG))
print("rebuilt HH chunks :", hh_new.count())

tv = table_stats(spark, C.TBL_CHUNKS)
print(C.TBL_CHUNKS, "-> rows:", tv["n_rows"], "| size:", bytes_human(tv["size_bytes"]))